# BrandSphere AI — Week 10: Integration Tests
**CRS AI Capstone 2025–26**

Tests the full end-to-end pipeline:
1. Brand inputs → palette generation
2. Palette → logo generation
3. Logo → animation
4. Brand inputs → slogan generation
5. Campaign prediction pipeline
6. Full brand kit ZIP export

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly nltk Pillow 2>/dev/null
import subprocess, os, sys
r = subprocess.run(['git','clone','--quiet','https://github.com/iblamehemer/og','/content/BrandSphere'],capture_output=True,text=True)
os.chdir('/content/BrandSphere'); sys.path.insert(0,'/content/BrandSphere')
os.makedirs('assets/sample_exports',exist_ok=True)
print('✅ Ready')

In [ ]:
import pandas as pd, numpy as np, pickle, json, os, io, zipfile
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'figure.facecolor':'#0C0D0F','axes.facecolor':'#141518','text.color':'white','axes.labelcolor':'white','xtick.color':'white','ytick.color':'white'})
import warnings; warnings.filterwarnings('ignore')

# Import all src modules
from src.config import INDUSTRIES, PERSONALITIES, TONES, PLATFORMS, COLOR_PSYCHOLOGY
from src.palette_engine import generate_palette, score_palette_harmony
from src.font_engine import recommend_fonts
from src.logo_engine import generate_all_logos, svg_to_png_bytes
from src.slogan_engine import generate_slogans, nltk_analyze
from src.animation_engine import create_brand_gif
print('✅ All modules imported successfully')

## Test 1: Brand Inputs Validation

In [ ]:
# Define test brand profile
test_brand = {
    'company':     'NovaTech Solutions',
    'industry':    'Technology / Software',
    'personality': 'Minimalist',
    'tone':        'Professional',
    'audience':    'CTOs and IT Directors at mid-market companies',
    'platform':    'LinkedIn',
    'region':      'North America',
}

print('Test brand profile:')
for k, v in test_brand.items():
    valid = v in (INDUSTRIES if k=='industry' else PERSONALITIES if k=='personality' 
                  else TONES if k=='tone' else PLATFORMS if k=='platform' else [v])
    status = '✅' if valid or k in ['company','audience','region'] else '❌'
    print(f'  {status} {k:12s}: {v}')

## Test 2: Palette Generation

In [ ]:
palette = generate_palette(test_brand['industry'], test_brand['personality'])
print(f'Palette generated: {len(palette)} colours')
for name, data in palette.items():
    print(f'  {name:15s}: {data["hex"]}')

harmony = score_palette_harmony(palette)
print(f'\nHarmony score: {harmony}/1.0  {"✅ Good" if harmony > 0.4 else "⚠️ Low"}')
assert len(palette) == 5, "Must produce exactly 5 colours"
assert harmony >= 0.0,    "Harmony must be non-negative"
print('✅ Test 2 PASSED')

## Test 3: Logo Generation

In [ ]:
logos = generate_all_logos(test_brand['company'], palette)
print(f'Logos generated: {len(logos)}')
for logo in logos:
    print(f'  Style {logo["index"]+1}: {logo["style"]:25s} — SVG: {len(logo["svg"])} chars')

# Test PNG export
png_bytes = svg_to_png_bytes(logos[0]['svg'], size=300)
print(f'\nPNG export: {len(png_bytes)//1024} KB')
assert len(logos) == 5,         "Must produce exactly 5 logos"
assert all(l['svg'] for l in logos), "All logos must have SVG content"
assert len(png_bytes) > 0,      "PNG export must succeed"
print('✅ Test 3 PASSED')

## Test 4: Font Recommendations

In [ ]:
fonts = recommend_fonts(test_brand['industry'], test_brand['personality'])
print(f'Font pairings: {len(fonts)}')
for f in fonts:
    print(f"  Rank {f['rank']}: {f['heading']} + {f['body']}")
    print(f"    Rationale: {f['rationale'][:70]}…")
    print(f"    URL: {f['google_url']}")

assert len(fonts) >= 1, "Must return at least 1 font pairing"
print('✅ Test 4 PASSED')

## Test 5: NLTK Slogan Analysis

In [ ]:
test_slogans = [
    'Engineering the future, one line at a time.',
    'Intelligence built in. Complexity built out.',
    'Where technology meets simplicity.',
]

print('NLTK analysis results:')
for slogan in test_slogans:
    result = nltk_analyze(slogan)
    print(f'\n  Slogan: "{slogan}"')
    for k, v in result.items():
        print(f'    {k}: {v}')

print('\n✅ Test 5 PASSED')

## Test 6: Animation Generation

In [ ]:
from PIL import Image
gif_bytes = create_brand_gif(logos[0]['svg'], test_slogans[0], palette, 
                               test_brand['company'], 'typewriter', frames=15)

# Verify GIF
assert gif_bytes[:3] == b'GIF', "Output must be a valid GIF"
img = Image.open(io.BytesIO(gif_bytes))
print(f'Animation: {len(gif_bytes)//1024} KB | Size: {img.size} | Mode: {img.mode}')

# Count frames
frame_count = 0
try:
    while True:
        frame_count += 1
        img.seek(frame_count)
except EOFError:
    pass
print(f'Frames: {frame_count}')
assert frame_count >= 10, "Must have at least 10 frames"
print('✅ Test 6 PASSED')

## Test 7: Campaign Prediction

In [ ]:
import sklearn
from sklearn.preprocessing import LabelEncoder, StandardScaler

with open('models/roi_model.pkl','rb') as f:
    roi_model = pickle.load(f)
with open('models/scaler.pkl','rb') as f:
    scaler = pickle.load(f)
with open('models/encoders.pkl','rb') as f:
    encoders = pickle.load(f)

# Build test feature vector
sample = np.array([[0, 2, 3, 0, 1, 30, 5000, 3.2, 1500, 46875, 3, 1, 3.33, 1562.5]])
sample_scaled = scaler.transform(sample)
prediction = roi_model.predict(sample_scaled)[0]

print(f'Campaign prediction test:')
print(f'  Model type:  {type(roi_model).__name__}')
print(f'  Input dims:  {sample.shape}')
print(f'  Predicted ROI: {prediction:.4f}')
assert isinstance(prediction, (float, np.floating)), "Prediction must be numeric"
print('✅ Test 7 PASSED')

## Test 8: Full Pipeline Summary

In [ ]:
print('=' * 55)
print('BRANDSPHERE AI — INTEGRATION TEST RESULTS')
print('=' * 55)

tests = [
    ('Brand Input Validation',   True),
    ('Palette Generation (K=5)', len(palette) == 5),
    ('Harmony Scoring',          harmony >= 0.0),
    ('Logo Generation (5 styles)', len(logos) == 5),
    ('PNG Export',               len(png_bytes) > 0),
    ('Font Recommendations',     len(fonts) >= 1),
    ('NLTK Slogan Analysis',     True),
    ('GIF Animation',            len(gif_bytes) > 0),
    ('Campaign Prediction',      True),
]

all_pass = True
for name, result in tests:
    status = '✅ PASS' if result else '❌ FAIL'
    print(f'  {status}  {name}')
    if not result: all_pass = False

print(f'\n{"✅ ALL TESTS PASSED" if all_pass else "❌ SOME TESTS FAILED"}')
print(f'BrandSphere AI is ready for deployment.')